# Extração mensal de documentos PNCP com Docling

Execute pelo navegador do Google Colab com GPU T4. Este notebook processa um lote lógico de até 1.000 PDFs em microlotes de 50, mantendo checkpoint após cada microlote.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
import sys
from pathlib import Path

# Ajuste somente se voce escolher outro nome para a pasta do projeto no Drive.
PROJETO_DRIVE = Path('/content/drive/MyDrive/pncp_projeto')
if not PROJETO_DRIVE.is_dir():
    raise FileNotFoundError(f'Projeto nao encontrado: {PROJETO_DRIVE}')

sys.path.insert(0, str(PROJETO_DRIVE))
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(PROJETO_DRIVE / 'requirements-colab.txt')],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU indisponivel. Selecione T4 GPU no menu Runtime.')
print('GPU em uso:', torch.cuda.get_device_name(0))


In [ ]:
# ---------------- CONFIGURACAO DO LOTE ----------------
ANO = 2026
MES = 8
LIMITE_DOCUMENTOS = 1_000  # lote logico; use None para todos os pendentes
TAMANHO_MICROLOTE = 50
PERFIL = 'qualidade'       # 'qualidade' agora; 'digital_rapido' futuramente
REPROCESSAR_ERROS = False

# O ZIP precisa ter original/ e expanded/ na raiz.
ARQUIVO_ZIP_DRIVE = Path(
    f'/content/drive/MyDrive/pncp_lotes/{ANO}_{MES:02d}/documentos.zip'
)
RAIZ_DOCUMENTOS_COLAB = Path('/content/pncp_documentos')

if not ARQUIVO_ZIP_DRIVE.is_file():
    raise FileNotFoundError(f'ZIP do lote nao encontrado: {ARQUIVO_ZIP_DRIVE}')


In [ ]:
import shutil
import zipfile

ZIP_LOCAL = Path('/content/documentos.zip')
shutil.copy2(ARQUIVO_ZIP_DRIVE, ZIP_LOCAL)
shutil.rmtree(RAIZ_DOCUMENTOS_COLAB, ignore_errors=True)
RAIZ_DOCUMENTOS_COLAB.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_LOCAL) as arquivo_zip:
    arquivo_zip.extractall(RAIZ_DOCUMENTOS_COLAB)

print('Documentos prontos em:', RAIZ_DOCUMENTOS_COLAB)


In [ ]:
from src.base_estruturada.pipelines.pipeline_documentos import (
    PeriodoColeta, caminhos_do_periodo,
)
from src.base_estruturada.pipelines.pipeline_docling_lotes import (
    executar_docling_por_lotes, gerar_fila_docling,
)
from src.base_estruturada.pncp.repository import (
    carregar_checkpoint_documentos, carregar_json,
    materializar_manifesto_com_checkpoint,
)

periodo = PeriodoColeta(ano=ANO, mes=MES)
caminhos = caminhos_do_periodo(periodo)

manifestos = {
    'contratos': materializar_manifesto_com_checkpoint(
        carregar_json(PROJETO_DRIVE / caminhos['manifesto_contratos'], padrao=[]),
        carregar_checkpoint_documentos(
            PROJETO_DRIVE / caminhos['checkpoint_documentos_contratos']
        ),
    ),
    'contratacoes': materializar_manifesto_com_checkpoint(
        carregar_json(PROJETO_DRIVE / caminhos['manifesto_contratacoes'], padrao=[]),
        carregar_checkpoint_documentos(
            PROJETO_DRIVE / caminhos['checkpoint_documentos_contratacoes']
        ),
    ),
}

fila = gerar_fila_docling(
    periodo=periodo.identificador,
    manifestos=manifestos,
    caminho_fila=PROJETO_DRIVE / caminhos['fila_extracao_docling'],
)
print('PDFs na fila:', fila['quantidadeDocumentos'])


In [ ]:
resumo = executar_docling_por_lotes(
    fila=fila,
    caminho_estado=PROJETO_DRIVE / caminhos['checkpoint_extracao_docling'],
    caminho_relatorio=PROJETO_DRIVE / caminhos['relatorio_extracao_docling'],
    raiz_documentos=RAIZ_DOCUMENTOS_COLAB,
    pasta_saida=PROJETO_DRIVE / caminhos['pasta_textos_docling'],
    raiz_projeto_portavel=PROJETO_DRIVE,
    tamanho_microlote=TAMANHO_MICROLOTE,
    limite_documentos=LIMITE_DOCUMENTOS,
    perfil=PERFIL,
    reprocessar_erros=REPROCESSAR_ERROS,
)
print('Resumo final:', resumo)
